# DAGonStar scientific-workflow tutorial

This notebook is the executable companion to the canonical Lessons 00–32. It is designed for a local Jupyter kernel and Google Colab. Lessons 01–16 execute deterministic local examples, including the FaaS, IoT, and agentic mocks; no model endpoint, device, or cloud function is contacted. Lessons 17–26 execute lesson scripts that mock the Docker, Kubernetes, Nomad, SSH, cloud, and scheduler clients or use portable emulation, because a hosted notebook cannot substantiate live infrastructure; a script whose optional client package is missing prints an explicit `skipped` line. Lessons 27–32 execute local FAIR, composition, CWL, continuum, operations, and capstone checks.

For each lesson, state a prediction before running the cell, retain the output as evidence, and distinguish that evidence from claims about external infrastructure or scientific reproducibility.


## Lesson 00 — Establish the environment

The setup cell locates an existing checkout. When the notebook is opened as a standalone file in Colab, it clones the public repository into `/content/dagonstar`. It installs the checkout with the optional Docker and Kubernetes Python clients (used only by the mocked Lessons 17 and 19) into the active kernel environment, then defines interpreter-safe helpers used by every later cell. The second cell reports the platform, registered task types, and optional packages.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys


COLAB_ROOT = Path(os.environ.get("DAGONSTAR_COLAB_ROOT", "/content/dagonstar"))


def _find_repository():
    candidates = [Path.cwd(), *Path.cwd().parents, COLAB_ROOT]
    for candidate in candidates:
        if (candidate / "dagon").is_dir() and (candidate / "examples" / "tutorial").is_dir():
            return candidate.resolve()
    return None


ROOT = _find_repository()
try:
    import google.colab  # type: ignore[import-not-found]
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
if ROOT is None:
    if not IN_COLAB:
        raise RuntimeError("Open this notebook from a DAGonStar checkout, or set ROOT to that checkout.")
    ROOT = COLAB_ROOT
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/DagOnStar/dagonstar.git", str(ROOT)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[docker,kubernetes]"],
    cwd=ROOT,
    check=True,
)

RUN_ENV = os.environ.copy()
RUN_ENV["PYTHONPATH"] = str(ROOT) + os.pathsep + RUN_ENV.get("PYTHONPATH", "")


def run_example(relative_path):
    completed = subprocess.run(
        [sys.executable, str(ROOT / relative_path)],
        cwd=ROOT,
        env=RUN_ENV,
        check=True,
    )
    print(f"PASS: {relative_path}")
    return completed


def run_tests(*test_names):
    completed = subprocess.run(
        [sys.executable, "-m", "unittest", "-v", *test_names],
        cwd=ROOT,
        env=RUN_ENV,
        check=True,
    )
    print("PASS:", ", ".join(test_names))
    return completed


print("runtime:", "Google Colab" if IN_COLAB else "Jupyter/local")
print("python:", sys.version.split()[0])
print("repository:", ROOT)


In [ ]:
run_example("examples/tutorial/lesson_00_environment_check.py")


## Lesson 01 — Create and run the first local task

Predict the output artifact and its value, then execute the credential-free local batch example.


In [ ]:
run_example("examples/tutorial/lesson_01_first_local_task.py")


## Lesson 02 — Build a directed acyclic graph

Predict the report task's two predecessors. The example verifies fan-out, fan-in, and completion order.


In [ ]:
run_example("examples/tutorial/lesson_02_build_a_dag.py")


## Lesson 03 — Connect tasks with workflow data references

Predict how the logical `workflow://` reference becomes a consumer-visible path and which dependency is discovered.


In [ ]:
run_example("examples/tutorial/lesson_03_workflow_data_dependencies.py")


## Lesson 04 — Validate and repair a cycle

Predict why the deliberate cycle has no topological order. The example verifies rejection and then validates the repaired graph without executing the invalid graph.


In [ ]:
run_example("examples/tutorial/lesson_04_validate_and_fix_cycles.py")


## Lesson 05 — Inspect scratch directories, launchers, and logs

Distinguish the scientific output from DAGonStar's operational records below `.dagon/` before running the example.


In [ ]:
run_example("examples/tutorial/lesson_05_scratch_launchers_and_logs.py")


## Lesson 06 — Compare data-staging modes

Hold the producer and consumer constant while predicting the filesystem difference between `COPY` and `LINK`.


In [ ]:
run_example("examples/tutorial/lesson_06_data_staging.py")


## Lesson 07 — Checkpoint and resume

Predict which completed task state can be reused and remember that checkpoint reuse does not validate undeclared external inputs.


In [ ]:
run_example("examples/tutorial/lesson_07_checkpoint_and_resume.py")


## Lesson 08 — Observe asynchronous lifecycle events

Predict the lifecycle event order. Joining the returned thread is required before treating the workflow as complete.


In [ ]:
run_example("examples/tutorial/lesson_08_asynchronous_execution.py")


## Lesson 09 — Execute a native Python function

Predict the structured JSON result produced by the importable Python target.


In [ ]:
run_example("examples/tutorial/lesson_09_native_python_tasks.py")


## Lesson 10 — Execute a web task against a local mock

The example binds a short-lived loopback server and makes no public request. Success demonstrates request construction and artifact handling, not production-service availability.


In [ ]:
run_example("examples/tutorial/lesson_10_web_tasks.py")


## Lesson 11 — Execute an LLM task against a local mock

The fixed local response isolates orchestration from provider cost, credentials, drift, and stochastic output. It does not evaluate a real model.


In [ ]:
run_example("examples/tutorial/lesson_11_llm_tasks.py")


## Lesson 12 — Invoke a serverless function through the FaaS mock

Predict the mock result and how many provider calls a checkpointed re-execution makes. The cell also checks that no cloud SDK was imported. It does not invoke a deployed function.


In [ ]:
run_example("examples/tutorial/lesson_12_faas_tasks.py")
run_tests("tests.test_faas")


## Lesson 13 — Observe and actuate devices with bounded IoT tasks

Predict which observation is rejected and how an unacknowledged actuation is recorded. The mock provider contacts no broker or device.


In [ ]:
run_example("examples/tutorial/lesson_13_iot_tasks.py")
run_tests("tests.test_iot_task")


## Lesson 14 — Specify and configure an agentic task

The example writes a temporary `dagon.ini` whose `quality-local` profile uses the deterministic `mock` agent provider. Predict the fit of each attempt and which four declarations are rejected before execution.


In [ ]:
run_example("examples/tutorial/lesson_14_agentic_specification.py")
run_tests("tests.test_agentic_config")


## Lesson 15 — Constrain agent tools with guardrails

Predict the guardrail code for an undeclared tool, a path escape, and a `workflow://` injection. The adversarial run is expected to fail; the cell passes only if the refused action had no side effect.


In [ ]:
run_example("examples/tutorial/lesson_15_agentic_tools_and_guardrails.py")
run_tests("tests.test_agentic_guardrails")


## Lesson 16 — Evaluate, publish, and checkpoint agent results

Compute F = (2·1 + 1·0)/3 before running. Predict whether the downstream task runs when a mandatory gate fails, and which changes invalidate the checkpoint.


In [ ]:
run_example("examples/tutorial/lesson_16_agentic_evaluation_and_checkpoints.py")
run_tests("tests.test_agentic_evaluation", "tests.test_agentic_checkpoint")


## Lesson 17 — Verify the Docker task boundary

The script mocks the Docker client; it verifies the `docker exec` wrapper and a remote endpoint on SSH port 2200. A passing cell is not evidence that a container ran.


In [ ]:
run_example("examples/tutorial/lesson_17_docker_tasks.py")
run_tests("tests.test_docker_task")


## Lesson 18 — Verify Apptainer image and execution commands

`subprocess.run` is mocked. Predict whether a prebuilt SIF triggers a build. Live Apptainer execution is optional and not run here.


In [ ]:
run_example("examples/tutorial/lesson_18_apptainer_tasks.py")
run_tests("tests.test_apptainer_task")


## Lesson 19 — Verify the Kubernetes pod lifecycle

The Kubernetes API is mocked; the cell prints a skip line if the `kubernetes` package is unavailable. No cluster is contacted.


In [ ]:
run_example("examples/tutorial/lesson_19_kubernetes_tasks.py")
run_tests("tests.test_kubernetes_task")


## Lesson 20 — Verify the Nomad job lifecycle

Predict the result for a terminal event with exit code 0, then read the tests for non-zero exit, missing allocation, and timeout. The HTTP API is mocked.


In [ ]:
run_example("examples/tutorial/lesson_20_nomad_tasks.py")
run_tests("tests.test_nomad_task")


## Lesson 21 — Verify the SSH task boundary

These checks verify default and custom SSH ports, quoting of remote commands, and SSH policy without opening a network connection.


In [ ]:
run_example("examples/tutorial/lesson_21_ssh_remote_tasks.py")
run_tests("tests.test_security_configuration", "tests.test_shell_commands")


## Lesson 22 — Configure a cloud task without provisioning

Predict whether constructing the task imports Apache Libcloud. Nothing is provisioned; the provider import is simulated as missing.


In [ ]:
run_example("examples/tutorial/lesson_22_cloud_tasks.py")
run_tests("tests.test_optional_integrations")


## Lesson 23 — Verify the Slurm task boundary

The script quotes hostile scheduler values and runs a succeeding and a failing job through a local mock `sbatch`. The tests inspect command generation and BeeOND fragments. No job reaches a real scheduler.


In [ ]:
run_example("examples/tutorial/lesson_23_slurm_tasks.py")
run_tests(
    "tests.test_optional_integrations.OptionalIntegrationTests.test_slurm_command_includes_configured_scheduler_options",
    "tests.test_optional_integrations.OptionalIntegrationTests.test_slurm_command_quotes_all_user_controlled_values",
    "tests.test_beeond.SlurmCommandTests",
    "tests.test_beeond.LauncherExecutionTests.test_task_volume_is_persisted_and_verified",
)



## Lesson 24 — Use BeeOND job-local storage under portable emulation

Predict which BeeOND options are rejected and where a task volume is persisted. Emulation does not reproduce volume speed or multi-node sharing.

The second part binds DAGonFS, BeeOND, Hercules, CAPIO, and DynoStore through the common ad-hoc interface. Predict the `DAGON_ADHOC_ACCESS` value of each provider before running it; no provider is started under emulation.

The third part adds a task-scope volume next to a default workflow volume through an adapter written outside the package, then streams frames from B1 to B2 while C and D keep ordinary staging. Predict which task starts before its producer has finished.


In [ ]:
run_example("examples/tutorial/lesson_24_beeond_storage.py")
run_example("examples/dataflow/slurm/beeond_local_emulation.py")
run_example("examples/adhoc/portable.py")
run_example("examples/adhoc/custom_adapter.py")
run_example("examples/adhoc/streaming.py")
run_example("examples/adhoc/task_scope_volume.py")
run_tests("tests.test_adhoc_filesystems")
run_tests("tests.test_adhoc_streaming")


## Lesson 25 — Stage inputs before scheduler submission

Predict whether `sbatch` runs when a referenced input is missing. The script uses the local mock scheduler.


In [ ]:
run_example("examples/tutorial/lesson_25_scheduler_prestaging.py")


### Scheduler-aware pre-staging: legacy versus enabled ordering

By default, a Slurm job stages its `workflow://` inputs after its cores are allocated. With `Workflow(pre_stage=True)`, or `pre_stage=True` on one task, inputs are staged and verified on the submission host before `sbatch`. Where the source is readable there, the staged content is compared with it, and a completed task is reused on resume only if its inputs are unchanged. The example below runs one workflow in both lifecycles. A mock `sbatch` records whether the stage-in marker and the inputs existed at submission time. No Slurm and no secrets are required; queue times are not emulated.


In [ ]:
run_example("examples/dataflow/slurm/pre_stage_mock.py")
run_tests(
    "tests.test_scheduler_prestaging.LifecycleOrderTests",
    "tests.test_scheduler_prestaging.LocalMockSchedulerTests.test_missing_input_fails_staging_without_submission",
    "tests.test_scheduler_prestaging.LocalMockSchedulerTests.test_corrupted_transfer_is_not_marked_complete",
    "tests.test_scheduler_prestaging.LocalMockSchedulerTests.test_changed_input_invalidates_a_successful_checkpoint",
)


**Exercise.** A job requests 200 cores and its inputs take 30 minutes to stage. Use the cell below to compute the allocated core-hours the legacy lifecycle holds idle during stage-in. Then explain why this figure estimates allocated resources not held, and does not measure a queue-time saving.


In [ ]:
from dagon.prestage import estimated_core_hours_saved

saved = estimated_core_hours_saved(200, 30 * 60)
print("Allocated core-hours not held during stage-in:", saved)
assert saved == 100.0


## Lesson 26 — Select data movers and transfer services

Predict the mover for same-host, SCP-capable, and Globus-endpoint tasks. Remote tasks and the Globus manager are mocked; no data crosses a network.


In [ ]:
run_example("examples/tutorial/lesson_26_data_movers.py")
run_tests("tests.test_staging", "tests.test_dynostore_examples")


## Lesson 27 — Record FAIR metadata locally

The lesson script checks that FAIR recording is opt-in and that a configuration secret reaches no exported file; the FAIR example verifies the nine local exports. Nothing is published to a repository.


In [ ]:
run_example("examples/tutorial/lesson_27_fair_by_design.py")
run_example("examples/fair/fair_local.py")
run_tests("tests.test_fair")


## Lesson 28 — Compose meta-workflows

The script composes two workflows with `DAG_TPS`, resolves a cross-workflow `workflow://` reference, and shows the failure without the optional service. It then runs a `TaskType.META` task that calls an external, engine-free workflow through the `command` adapter and exchanges data only through `workflow://` references.


In [ ]:
run_example("examples/tutorial/lesson_28_meta_workflows.py")
run_tests("tests.test_workflow_core.WorkflowCoreTests.test_transversal_references_resolve_across_loaded_workflows")
run_tests("tests.test_meta_task.MetaExecutionTests")


## Lesson 29 — Export the command graph as CWL v1.2

The script checks deterministic export and runs `cwltool --validate` when it is installed; otherwise it prints that validation was skipped. The agentic example executes an exported agent step with `cwltool` only when both tools are present.

The streaming export turns `workflow:///produce/items?stream` into a `Directory` passed from producer to consumer and keeps the intent in a `dagon:Stream` hint, because CWL v1.2 runs steps one after the other.


In [ ]:
run_example("examples/tutorial/lesson_29_cwl_export.py")
run_example("examples/agentic/cwl_local.py")
run_example("examples/cwl/export_streaming_workflow.py")
run_tests("tests.test_cwl_example")


## Lesson 30 — Operate across the compute continuum

Predict the task types in order and whether the FaaS step runs if the agent result were rejected. All providers are mocks.


In [ ]:
run_example("examples/tutorial/lesson_30_compute_continuum.py")
run_example("examples/iot/observe_mock.py")


## Lesson 31 — Operate, deploy, and troubleshoot

The DAGon service API is mocked. The cell also checks the runner environment and, when installed, the `dagon-agent-runner` console script.


In [ ]:
run_example("examples/tutorial/lesson_31_operations.py")
run_tests("tests.test_runtime_env", "tests.test_platform_support")


## Lesson 32 — Capstone: every task type with evidence

Predict how many task types the capstone contains. Any live adapter or network call would fail the cell. Portable emulation is not evidence of infrastructure behaviour.


In [ ]:
run_example("examples/tutorial/lesson_32_capstone.py")


## Course verification

A complete run should leave every executable cell successful. The validators check the canonical lesson structure, cross-links, notebook coverage, and the mapping of every task type and inventoried feature to lessons, examples, and tests. A pass supports only the local, mocked, and emulated claims stated above.


In [ ]:
run_example("tools/validate_tutorial.py")
run_example("tools/validate_tutorial_coverage.py")
